### Test AI Agent 🤖 Tool Calling with DeepEval 🧪

Testing AI Agent involves testing of the Tools being invoked by an AI Agent. Here, AI Agent will invoke the necessary tools based on the given input and respond with the help of the tools being bounded with the AI Agent


<img src="./img/AIAGent.png" width="800" height="400" style="display: block; margin: auto;">

In [1]:
!pip install -qU ddgs

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

#### AI Agent with Tools

In [3]:
from langchain.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

@tool
def add_numbers(a: int, b: int) -> int:
    "Add two numbers and return results."
    return int(a) + int(b)

@tool
def subtract_numbers(a: int, b: int) -> int:
    "Subtract two numbers and return results."
    return int(a) - int(b)

tools = [add_numbers, subtract_numbers, search_tool]

agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    return_intermediate_steps=True
)

response = agent.invoke("Who is the current president of USA in 2025, just give the name")

print(response)

def query_ai_agent(question):
    response = agent.invoke(question)
    intermediate_steps = response['intermediate_steps']
    agent_action, results = intermediate_steps[0]
    tool = agent_action.tool
    tool_input = agent_action.tool_input
    return response,tool, tool_input,


/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_62828/579676457.py:19: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Thought: I need to search for the current president of USA in 2025.

Action:
```
{
  "action": "duckduckgo_search",
  "action_input": "current president of USA in 2025"
}
```
Observation: 1 day ago - Trump also fired members of the Equal Employment Opportunity Commission, National Labor Relations Board and 56 senior officials at USAID for allegedly attempting to thwart his priorities. Trump issued NSPM-7 (National Security Presidential Memorandum-7) on September 25, 2025, directing a ... 2 weeks ago - In addition, nine vice presidents have become president by virtue of a president's intra-term death or resignation. In all, 45 individuals have served, or are serving, 47 presidencies as of the current 60th four-year term. Donald Trump is the 47th and current president since January 20, 2025. 7 hours ago - Biden entered office with majority support, but his approval ratings declined significantly during his presidency, particularly over concerns abo

In [4]:
response,tool, tool_input = query_ai_agent("Who is the president of USA in 2025, just give me the name")

print(response)

print(tool)

print(tool_input)



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "duckduckgo_search",
  "action_input": "who will be the president of USA in 2025"
}
```
Observation: 1 day ago - Donald Trump's second and current tenure as the president of the United States began upon his inauguration as the 47th president on January 20, 2025. Trump, a Republican, previously served as the 45th president from 2017 to 2021. He lost re-election to Democratic nominee Joe Biden in 2020, ... 2 weeks ago - The incumbent president is Donald Trump, who assumed office on January 20, 2025. Since the office was established in 1789, 45 men have served in 47 presidencies. The discrepancy is due to the nonconsecutive terms of Grover Cleveland (counted ... 2 weeks ago - In addition, nine vice presidents have become president by virtue of a president's intra-term death or resignation. In all, 45 individuals have served, or are serving, 47 presidencies as of the current 60th four-year term. Donald Trump is the ... 6 hou

### Testing AI Agent with DeepEval

In [5]:
import deepeval

deepeval.login("chf7LtTWtK1foTOAiK+vHFZ622I16kZtcpzfC+7FAVU=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [6]:
!deepeval set-ollama --model qwen2.5:latest

🙌 Congratulations! You're now using a local Ollama model `qwen2.5:latest` for 
all evals that require an LLM.


In [7]:
from deepeval.test_case import ToolCall

test_data = [
    {
        "input": "What is the sum of 20 and 40",
        "expected_output": "60",
        "tool_called": [
            ToolCall(name = "add_numbers")
        ]
    },
     {
        "input": "Who is the president of USA in 2025, just give me the name",
        "expected_output": "Donald Trump",
        "tool_called": [
            ToolCall(name = "duckduckgo_search")
        ]
    }
]

In [8]:
test_data

[{'input': 'What is the sum of 20 and 40',
  'expected_output': '60',
  'tool_called': [ToolCall(
       name="add_numbers"
   )]},
 {'input': 'Who is the president of USA in 2025, just give me the name',
  'expected_output': 'Donald Trump',
  'tool_called': [ToolCall(
       name="duckduckgo_search"
   )]}]

In [9]:
test_data[0]['tool_called']

[ToolCall(
     name="add_numbers"
 )]

In [11]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ToolCorrectnessMetric

test_cases = []
for testcase in test_data:
  response,tool, tool_input = query_ai_agent(testcase['input'])
  test_case = LLMTestCase(
    input=testcase['input'],
    tools_called=[ToolCall(name=tool)],
    actual_output=response["output"],
    expected_tools=testcase['tool_called']
  )

  test_cases.append(test_case)





> Entering new AgentExecutor chain...
Action:
```
{
  "action": "add_numbers",
  "action_input": {
    "a": 20,
    "b": 40
  }
}
```
Observation: 60
Thought:Action:
```
{
  "action": "Final Answer",
  "action_input": "The sum of 20 and 40 is 60."
}
```

> Finished chain.


> Entering new AgentExecutor chain...
Action:
```
{
  "action": "duckduckgo_search",
  "action_input": "who will be the president of USA in 2025"
}
```
Observation: The inauguration of Donald Trump as the 47th president of the United States took place on Monday, January 20, 2025. Due to freezing temperatures and high winds, it was held inside After a landslide election victory in 2024, President Donald J. Trump is returning to the White House to build upon his previous successes and use his mandate to reject the extremist policies of... President Donald Trump (R) was inaugurated for his second term as the 47th president of the United States on January 20, 2025. He serves with Vice President J.D. Vance (R), a forme

In [12]:
test_cases

[LLMTestCase(input='What is the sum of 20 and 40', actual_output='The sum of 20 and 40 is 60.', expected_output=None, context=None, retrieval_context=None, metadata=None, tools_called=[ToolCall(
     name="add_numbers"
 )], comments=None, expected_tools=[ToolCall(
     name="add_numbers"
 )], token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='Who is the president of USA in 2025, just give me the name', actual_output='Donald J. Trump', expected_output=None, context=None, retrieval_context=None, metadata=None, tools_called=[ToolCall(
     name="duckduckgo_search"
 )], comments=None, expected_tools=[ToolCall(
     name="duckduckgo_search"
 )], token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom

In [21]:
from deepeval.models import OllamaModel

metrics = ToolCorrectnessMetric(
    model=OllamaModel(model="qwen2.5:latest")
 )

for testcase in test_cases:
    metrics.measure(test_case=testcase)
    print(metrics.score)
    print(metrics.reason)
    print(testcase.expected_tools)

1.0
[
	 Tool Calling Reason: All expected tools ['add_numbers'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="add_numbers"
)]


1.0
[
	 Tool Calling Reason: All expected tools ['duckduckgo_search'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="duckduckgo_search"
)]
